In [30]:
import pandas as pd
import numpy as np
from pathlib import Path
import re
import json
import shutil

In [31]:
DATA_DIR = Path("/Volumes/Extension0/Graduate/100.MAIN/01.DATA/1.Training/LABEL")
PHOTO_DIR = Path("/Volumes/Extension0/Graduate/100.MAIN/01.DATA/1.Training/PHOTO")
DATA_VAL_DIR = Path("/Volumes/Extension0/Graduate/100.MAIN/01.DATA/2.Validation/LABEL")
PHOTO_VAL_DIR = Path("/Volumes/Extension0/Graduate/100.MAIN/01.DATA/2.Validation/PHOTO")

DIR_REGEX = re.compile('[A-Z]+_3.상추[0-9]+')

REF_PHOTO_DIR = Path("/Users/mainframe/Workspace/Graduate/dataset/train/photos")
REF_LABEL_DIR = Path("/Users/mainframe/Workspace/Graduate/dataset/train/labels")
REF_PHOTO_VAL_DIR = Path("/Users/mainframe/Workspace/Graduate/dataset/validate/photos")
REF_LABEL_VAL_DIR = Path("/Users/mainframe/Workspace/Graduate/dataset/validate/labels")

MIN_DIR = 54
MAX_DIR = 75
MIN_VAL_DIR = 7
MAX_VAL_DIR = 10

In [71]:
def extract_photos_from_src_dir(src_dir, dst_dir, min_no, max_no):
    list_of_photos = []
    
    #Move Every Photos that has good portrait.
    for entry in src_dir.iterdir():
        if entry.is_dir() and DIR_REGEX.match(entry.name):
            dir_no = int(entry.name.split('상추')[1])
            if min_no <= dir_no and dir_no <= max_no:
                for subentry in entry.iterdir():
                    if subentry.is_file() and subentry.name != ".DS_Store":
                        list_of_photos.append(subentry)
                        shutil.copy(subentry, dst_dir / subentry.name)
    
    return list_of_photos

In [33]:
list_of_photos = extract_photos_from_src_dir(PHOTO_DIR, REF_PHOTO_DIR, MIN_DIR, MAX_DIR)

In [72]:
list_of_photos_val = extract_photos_from_src_dir(PHOTO_VAL_DIR, REF_PHOTO_VAL_DIR, MIN_VAL_DIR, MAX_VAL_DIR)

In [55]:
def move_labels_from_src_dir(src_dir, dst_dir):
    #Perform and exhuastive search, and yes this is intentional.
    for entry in src_dir.iterdir():
        if entry.is_dir() and DIR_REGEX.match(entry.name):
            for subentry in entry.iterdir():
                if subentry.is_file() and subentry.name != ".DS_Store":
                    #print(f"Moving {subentry} to {REF_PHOTO_DIR / subentry.name}")
                    shutil.copy(subentry, dst_dir / subentry.name)

In [56]:
move_labels_from_src_dir(DATA_DIR, REF_LABEL_DIR)
move_labels_from_src_dir(DATA_VAL_DIR, REF_LABEL_VAL_DIR)

In [57]:
list_of_photos[0:5]

[PosixPath('/Volumes/Extension0/Graduate/100.MAIN/01.DATA/1.Training/PHOTO/TS_3.상추62/C30_L01_06_003_756540.jpg'),
 PosixPath('/Volumes/Extension0/Graduate/100.MAIN/01.DATA/1.Training/PHOTO/TS_3.상추62/C30_L01_06_001_778768.jpg'),
 PosixPath('/Volumes/Extension0/Graduate/100.MAIN/01.DATA/1.Training/PHOTO/TS_3.상추62/C30_L01_06_003_756554.jpg'),
 PosixPath('/Volumes/Extension0/Graduate/100.MAIN/01.DATA/1.Training/PHOTO/TS_3.상추62/C30_L01_06_001_855585.jpg'),
 PosixPath('/Volumes/Extension0/Graduate/100.MAIN/01.DATA/1.Training/PHOTO/TS_3.상추62/C30_L01_06_001_806877.jpg')]

In [58]:
list_of_photos_val[0:5]

[PosixPath('/Volumes/Extension0/Graduate/100.MAIN/01.DATA/2.Validation/PHOTO/VS_3.상추9/C46_L01_02_004_641843.jpg'),
 PosixPath('/Volumes/Extension0/Graduate/100.MAIN/01.DATA/2.Validation/PHOTO/VS_3.상추9/C46_L01_03_001_646122.jpg'),
 PosixPath('/Volumes/Extension0/Graduate/100.MAIN/01.DATA/2.Validation/PHOTO/VS_3.상추9/C46_L01_03_005_646177.jpg'),
 PosixPath('/Volumes/Extension0/Graduate/100.MAIN/01.DATA/2.Validation/PHOTO/VS_3.상추9/C46_L01_02_002_781265.jpg'),
 PosixPath('/Volumes/Extension0/Graduate/100.MAIN/01.DATA/2.Validation/PHOTO/VS_3.상추9/C46_L01_03_004_644334.jpg')]

In [59]:
def safe_avg_of(data, column_name):
    column = [e[column_name] for e in data]
    if None in column:
        return None
    num_column = [float(n) for n in column]
    return sum(num_column) / len(num_column)

In [60]:
def growth_to_number(growth):
    if growth == '정식기':
        return 1
    elif growth == '생육기':
        return 2
    elif growth == '수확기':
        return 3
    raise ValueError("Some other value has been detected")

In [66]:
def parse_json_file(file_path):
    """Reads a single JSON file and returns a dictionary or None if failed."""
    try:
        # Using pathlib's read_text handles the file opening/closing for you
        #dir_no, file_path = file_info
        raw_data = file_path.read_text(encoding='utf-8')
        
        data = json.loads(raw_data)

        leaf_ids = []
        stem_ids = []

        for cat in data["categories"]:
            if cat["name"] == "잎":
                leaf_ids.append(cat["id"])
            else:
                stem_ids.append(cat["id"])

        environment = data["envrionments"]
        
        extracted = {
            #'dir_no': dir_no,
            'file_name': file_path.name.split(".")[0],
            'growth': growth_to_number(data["images"]["growth_stage"]),
            'aa': data["images"]["growth_stage"],
            'captured': data["images"]["date_captured"],
            'kind': data["images"]["kind_type"],
            'temp': safe_avg_of(environment, "ti_value"),
            'humid': safe_avg_of(environment, "hi_value"),
            'co2': safe_avg_of(environment, "ci_value"),
            'light': safe_avg_of(environment, "ir_value"),
            'hyd_temp': safe_avg_of(environment, "tl_value"),
            'hyd_ec': safe_avg_of(environment, "ei_value"),
            'hyd_ph': safe_avg_of(environment, "pl_value"),
            'stem_area': sum([box["area"] if box["id"] in stem_ids else 0 for box in data["annotations"]]),
            'leaf_area': sum([box["area"] if box["id"] in leaf_ids else 0 for box in data["annotations"]]),
        }
        
        return extracted 
    except (json.JSONDecodeError, IOError) as e:
        print(f"Error parsing {file_path}: {e}")
    except TypeError as e:
        print(f"Error parsing {file_path}: {e}")
        return None

In [67]:
parsed_records = [parse_json_file(REF_LABEL_DIR / f"{photo_path.name.split(".")[0]}.json") for photo_path in list_of_photos]
df = pd.DataFrame(parsed_records)

In [68]:
df.head()

,file_name,growth,captured,kind,temp,humid,co2,light,hyd_temp,hyd_ec,hyd_ph,stem_area,leaf_area
0,C30_L01_06_003_756540,2,2022-01-08 02:16:24,로메인,20.0,81.0,267.5,79.0,20.70,0.05,7.4,273729.24,234047.59
1,C30_L01_06_001_778768,2,2022-01-11 16:16:24,로메인,18.0,82.5,369.5,0.0,19.90,0.05,NaN,325292.96,297204.99
2,C30_L01_06_003_756554,2,2022-01-08 06:16:24,로메인,19.0,78.5,247.0,87.0,20.65,0.05,7.5,282746.62,240854.46
3,C30_L01_06_001_855585,2,2022-01-21 22:16:24,로메인,19.0,74.5,1096.5,67.0,19.80,0.04,7.0,861586.26,334361.66
4,C30_L01_06_001_806877,2,2022-01-15 16:16:24,로메인,18.0,85.5,323.5,0.0,19.50,0.04,7.5,477949.56,317834.15


In [73]:
parsed_records_val = [parse_json_file(REF_LABEL_VAL_DIR / f"{photo_path.name.split(".")[0]}.json") for photo_path in list_of_photos_val]
df_val = pd.DataFrame(parsed_records_val)

In [74]:
df_val.head()

,file_name,growth,captured,kind,temp,humid,co2,light,hyd_temp,hyd_ec,hyd_ph,stem_area,leaf_area
0,C46_L01_02_004_641843,1,2021-10-06 12:49:11,로메인,23.5,76.0,474.0,29.0,23.35,0.08,6.50,91519.02,77914.99
1,C46_L01_03_001_646122,2,2021-12-02 06:19:03,로메인,20.0,22.0,473.5,79.0,18.75,0.09,6.10,1876939.90,1311398.16
2,C46_L01_03_005_646177,2,2021-12-02 11:19:03,로메인,18.0,24.0,469.0,44.5,17.90,0.09,6.50,1339388.21,979463.15
3,C46_L01_02_002_781265,1,2021-10-04 07:36:22,로메인,21.5,70.5,535.5,0.0,21.75,0.08,7.45,53361.75,40811.41
4,C46_L01_03_004_644334,1,2021-11-17 05:19:03,로메인,20.0,30.0,514.0,49.0,18.70,0.07,6.65,44733.04,28177.18


In [75]:
output_file = Path("plant_gotchi_train.parquet")
output_file_val = Path("plant_gotchi_validate.parquet")

# We use 'pyarrow' as the engine and 'snappy' for fast compression
df.to_parquet(
    output_file, 
    engine='pyarrow', 
    compression='snappy',
    index=False
)

print(f"Dataset(Train) successfully saved to: {output_file.absolute()}")

df_val.to_parquet(
    output_file_val, 
    engine='pyarrow', 
    compression='snappy',
    index=False
)

print(f"Dataset(Validate) successfully saved to: {output_file_val.absolute()}")

Dataset(Train) successfully saved to: /Users/mainframe/Workspace/Graduate/notebooks/plant_gotchi_train.parquet
Dataset(Validate) successfully saved to: /Users/mainframe/Workspace/Graduate/notebooks/plant_gotchi_validate.parquet
